In [2]:
# velo reward computation for rollouts on game24


"""Setup: read eval_rollout.jsonl. No model, no GPU.

`script/run_game24_one.py --score-vt` (default) augments each rollout row
with R_T, R_per_token, and a fixed-grid cumR_resampled. All three figures
below run from those fields plus the original (numbers, correct, n_tokens,
completion, global_step) columns.
"""
import json
from pathlib import Path
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Point this at the output directory produced by script/run_game24_one.py
RUN_DIR  = Path("logs/game24_sweep_exp4/len512/Qwen__Qwen3-0.6B")
EVAL_LOG = RUN_DIR / "eval_rollout.jsonl"
assert EVAL_LOG.exists(), f"missing {EVAL_LOG}; run script/run_game24_one.py first"

rows = [json.loads(l) for l in EVAL_LOG.read_text().splitlines() if l.strip()]
eval_df = pd.DataFrame(rows)
eval_df["key"] = eval_df["numbers"].apply(lambda x: tuple(sorted(x)))

has_vt = "R_T" in eval_df.columns and eval_df["R_T"].notna().any()
print(f"{len(eval_df)} eval rollouts across "
      f"{eval_df.global_step.nunique()} eval cycles "
      f"(global_step ∈ {sorted(eval_df.global_step.unique().tolist())})")
print(f"R_T fields present: {has_vt}")
eval_df.head(3)

4032 eval rollouts across 7 eval cycles (global_step ∈ [0, 200, 400, 600, 800, 1000, 1200])
R_T fields present: True


,step,idx,numbers,completion,expr,correct,n_tokens,n_cot_tokens,has_answer_marker,has_think_close,split,global_step,R_T,R_per_token,cumR_resampled,key
0,0,0,"[3, 5, 8, 9]","<think>\nOkay, let's see. I have the numbers 3...",,False,512,512,False,False,eval,0,-15.203041,-0.029693,"[-5.5611114501953125, -3.905657604487258, -12....","(3, 5, 8, 9)"
1,0,1,"[3, 5, 8, 9]","<think>\nOkay, let's see. The numbers given ar...",,False,512,512,False,False,eval,0,-9.174152,-0.017918,"[-5.5611114501953125, -3.905657604487258, -16....","(3, 5, 8, 9)"
2,0,2,"[3, 5, 8, 9]","<think>\nOkay, let's see. I need to use 3, 5, ...",,False,512,512,False,False,eval,0,-4.462837,-0.008716,"[-5.5611114501953125, -3.905657604487258, -4.4...","(3, 5, 8, 9)"


In [69]:
scorer_model = "Qwen/Qwen3-0.6B"
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok_src = scorer_model
# =============================
# Load scorer model & tokenizer
# =============================
tokenizer = AutoTokenizer.from_pretrained(tok_src)
device = (f"cuda:{torch.cuda.current_device()}"
                if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
# scorer = AutoModelForCausalLM.from_pretrained(
#     scorer_model, dtype=dtype
# ).to(device).eval()

# ==================================================
# Compute Velocity reward on a specific step & idx
# ==================================================
from script.rescore_vt import enumerate_solutions, to_chat

step = 1200
idx = 0
select_rows = [r for r in rows if r['global_step'] == step and r['idx'] == idx and r['expr']]
assert len(select_rows) > 0, f"No row found for step {step} and idx {idx}"
row = select_rows[0]

prompts, completions, refs = [], [], []
sols = enumerate_solutions(tuple(row["numbers"]))
puzzle = {"numbers": list(numbers), "solutions": sols}
prompt = tokenizer.apply_chat_template(
    to_chat(puzzle)["prompt"], tokenize=False, add_generation_prompt=True)
answer = row['expr']
for sol in sols: 
    prompts.append(prompt)
    completions.append(row["completion"])
    refs.append(sol)

In [ ]:
from src.velocity import compute_vt_batched

# ============================
# Velocity Reward Computation
# ============================

scored = compute_vt_batched(prompts, completions, refs, scorer, tokenizer,
                            micro_batch_size=8)
 

In [ ]:
# ====================================
# Visualize Velocity Reward across solutions
# - left:  cumulative R(t) = logp(ref | q, o_<t) − logp(ref | q) per ref
# - right: final R_T per ref, sorted; model's own answer highlighted
# ====================================
import numpy as np
import matplotlib.pyplot as plt

own = row["expr"]
own_idx = refs.index(own) if own in refs else None
R_T = np.array([s["R_T"] for s in scored])
order = np.argsort(R_T)[::-1]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4),
                               gridspec_kw={"width_ratios": [3, 2]})

for i, s in enumerate(scored):
    cum = s["logps"] - s["logps"][0]
    is_own = (i == own_idx)
    ax0.plot(cum, color="C3" if is_own else "0.6", lw=2 if is_own else 1,
             alpha=1.0 if is_own else 0.5,
             label=f"own ({'correct' if row['correct'] else 'wrong'}): {own}" if is_own else None)
ax0.axhline(0, color="k", lw=0.5)
ax0.set_xlabel("CoT prefix length t"); ax0.set_ylabel("cum R(t)")
ax0.set_title(f"R(t) over {len(scored)} refs  ·  step={row['global_step']}  idx={row['idx']}")
if own_idx is not None: ax0.legend(loc="best", fontsize=8)

colors = ["C3" if i == own_idx else "0.6" for i in order]
ax1.barh(range(len(order)), R_T[order], color=colors)
ax1.invert_yaxis()
ax1.set_yticks(range(len(order)))
ax1.set_yticklabels([refs[i] + (" ←own" if i == own_idx else "") for i in order],
                    fontsize=7)
ax1.set_xlabel("R_T"); ax1.set_title("final R_T, sorted")
plt.tight_layout(); plt.show()

if own_idx is not None:
    rank = int((R_T > R_T[own_idx]).sum())
    print(f"own answer rank: {rank}/{len(R_T)-1}   "
          f"R_T(own)={R_T[own_idx]:+.3f}   "
          f"R_T(best)={R_T.max():+.3f}   "
          f"gap={R_T.max() - R_T[own_idx]:+.3f}")


NameError: name 'top_mistake' is not defined

In [ ]:
# Case analysis
# (I). one instance of a CoT for a "correct rollout"
# Okay, let's see. I need to use each number exactly once (2,2,5,6) with +, -, *, /, and parentheses to make 24. Let's think of combinations.

# First, maybe combine some numbers. Let's try adding some. 2 + 2 = 4, then 6 - 5 = 1, but that doesn't help. Maybe 5 * 2 = 10, then 6 - 2 = 4, but 10 + 4 = 14. Not enough.

# What about (5 - (2/2))? That gives 4. Then 6 + 4*2? Wait, 4*2 is 8, plus 6 gives 14 again.

# Alternatively, maybe 6*2 =12, 5-2=3, then 12*(3*3) no, but there's only one 3. Maybe 6*(5 - (2/2))? Let's compute 2/2 is 1, 5-1 is 4, 6*4=24! Yes! That uses all numbers once: 6,5,2,2. So the expression is 6*(5 - (2/2)).

# Let me check the numbers: 6,5,2,2. Yes, each used once. So the expression is (6*(5 - (2/2))) which equals 24. So the final answer is that expression.

# -> we got the model trying and failing, it's not clear how useful this type of rollouts are to the success?
#    this CoT, whilst leading to correct answer, contains many errors\
#    "(5 - (2/2)) gives 4, then 6 + 4*2 wrongly uses 2 for three times --- only two 2s are provided"
#    "6*2 = 12, 5-2=3 then 12*(3*3) wrongly uses 3 twich despite it only appears once"
#    if we actually train the model on sth like this, such wrong and wierd reasoning will be baked into the model
#    and the real "sparkle" --- "(5 - (2/2))" -> "6*(5 - (2/2))" appears rather from analogy, not logical reduction
#    so we are not actually distilling the right thing here.



# print("Okay, let's see. I need to use each number exactly once (2,2,5,6) with +, -, *, /, and parentheses to make 24. Let's think of combinations.\n\nFirst, maybe combine some numbers. Let's try adding some. 2 + 2 = 4, then 6 - 5 = 1, but that doesn't help. Maybe 5 * 2 = 10, then 6 - 2 = 4, but 10 + 4 = 14. Not enough.\n\nWhat about (5 - (2/2))? That gives 4. Then 6 + 4*2? Wait, 4*2 is 8, plus 6 gives 14 again.\n\nAlternatively, maybe 6*2 =12, 5-2=3, then 12*(3*3) no, but there's only one 3. Maybe 6*(5 - (2/2))? Let's compute 2/2 is 1, 5-1 is 4, 6*4=24! Yes! That uses all numbers once: 6,5,2,2. So the expression is 6*(5 - (2/2)).\n\nLet me check the numbers: 6,5,2,2. Yes, each used once. So the expression is (6*(5 - (2/2))) which equals 24. So the final answer is that expression.")

Okay, let's see. I need to use each number exactly once (2,2,5,6) with +, -, *, /, and parentheses to make 24. Let's think of combinations.

First, maybe combine some numbers. Let's try adding some. 2 + 2 = 4, then 6 - 5 = 1, but that doesn't help. Maybe 5 * 2 = 10, then 6 - 2 = 4, but 10 + 4 = 14. Not enough.

What about (5 - (2/2))? That gives 4. Then 6 + 4*2? Wait, 4*2 is 8, plus 6 gives 14 again.

Alternatively, maybe 6*2 =12, 5-2=3, then 12*(3*3) no, but there's only one 3. Maybe 6*(5 - (2/2))? Let's compute 2/2 is 1, 5-1 is 4, 6*4=24! Yes! That uses all numbers once: 6,5,2,2. So the expression is 6*(5 - (2/2)).

Let me check the numbers: 6,5,2,2. Yes, each used once. So the expression is (6*(5 - (2/2))) which equals 24. So the final answer is that expression.
